In [1]:
# 02_model_building.ipynb
# Phase 7: TF-IDF features + stratified train/test split

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

print("pandas", pd.__version__, "| numpy", np.__version__)

pandas 3.0.5 | numpy 2.5.2


In [2]:
df = pd.read_csv("../data/processed/reviews_clean.csv")

print(df.shape)
print(df.columns.tolist())

(10254, 14)
['reviewerID', 'asin', 'reviewText', 'overall', 'unixReviewTime', 'reviewTime', 'helpful_votes', 'total_votes', 'review_date', 'sentiment', 'review_words', 'review_year', 'clean_review', 'clean_words']


In [3]:
# sanity: no missing/empty text, labels intact
print("NaN clean reviews   :", df["clean_review"].isna().sum())
print("empty clean reviews :", (df["clean_review"].astype(str).str.strip() == "").sum())
print()
print(df["sentiment"].value_counts())

NaN clean reviews   : 0
empty clean reviews : 0

sentiment
positive    9015
neutral      772
negative     467
Name: count, dtype: int64


In [4]:
X = df["clean_review"]   # the preprocessed text from Phase 6
y = df["sentiment"]      # negative / neutral / positive

print("X:", X.shape, "| y:", y.shape)

X: (10254,) | y: (10254,)


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 80% train / 20% test
    random_state=42,    # reproducible - same split every run
    stratify=y          # keep the 88 / 7.5 / 4.5 class ratios in BOTH sets
)

print("train:", X_train.shape[0], "reviews")
print("test :", X_test.shape[0], "reviews")

train: 8203 reviews
test : 2051 reviews


In [6]:
# class RATIOS should be nearly identical in all three columns
pd.DataFrame({
    "full":  y.value_counts(normalize=True).round(3),
    "train": y_train.value_counts(normalize=True).round(3),
    "test":  y_test.value_counts(normalize=True).round(3),
})

,full,train,test
sentiment,,,
positive,0.879,0.879,0.879
neutral,0.075,0.075,0.076
negative,0.046,0.046,0.045


In [7]:
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),   # unigrams + bigrams - EDA showed "not work" vs "works great"
    min_df=3,             # ignore terms appearing in fewer than 3 training reviews (typos, noise)
    max_features=20000,   # cap vocabulary size to keep the matrix manageable
)

# fit ONLY on training text - the test set must stay "unseen"
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)      # NOTE: transform, not fit_transform

print("train matrix:", X_train_tfidf.shape)
print("test matrix :", X_test_tfidf.shape)

train matrix: (8203, 20000)
test matrix : (2051, 20000)


In [8]:
feature_names = tfidf.get_feature_names_out()
print("vocabulary size:", len(feature_names))
print("some features:", feature_names[5000:5008])
print()

# did the bigrams we care about survive min_df=3?
for phrase in ["not work", "not good", "works great", "waste money", "easy use"]:
    print(f"{phrase!r:15} ->", phrase in tfidf.vocabulary_)

vocabulary size: 20000
some features: ['fold nice' 'fold nicely' 'fold pretty' 'fold small' 'folded' 'folding'
 'folding stand' 'folk']

'not work'      -> True
'not good'      -> True
'works great'   -> False
'waste money'   -> True
'easy use'      -> True


In [9]:
# how much of the matrix is actually non-zero?
density = X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1])
print(f"non-zero values: {X_train_tfidf.nnz:,}")
print(f"density: {density:.3%}")
print()

# ONE real review: which words/bigrams got the highest weights?
weights = X_train_tfidf.getrow(0).toarray().ravel()
nz = weights.nonzero()[0]

print("review text:", X_train.iloc[0][:120], "...")
print(f"\n{len(nz)} non-zero features out of {weights.size}\n")
pd.Series(weights[nz], index=feature_names[nz]).sort_values(ascending=False).head(10)

non-zero values: 406,751
density: 0.248%

review text: cheap look good also nice full sound holding well highly recommended ...

19 non-zero features out of 20000



nice full             0.348784
well highly           0.323067
also nice             0.323067
good also             0.319375
holding well          0.319375
full sound            0.312835
look good             0.241623
holding               0.225761
highly recommended    0.217317
full                  0.195580
dtype: float64

In [10]:
# low IDF = appears in almost every review (weak signal)
# high IDF = rare and specific (strong signal)
idf = pd.Series(tfidf.idf_, index=feature_names)

print("lowest IDF (in almost every review):")
print(idf.nsmallest(8).round(2))
print()
print("highest IDF (rare, very specific):")
print(idf.nlargest(8).round(2))

lowest IDF (in almost every review):
not       1.60
guitar    2.05
one       2.20
great     2.21
good      2.30
like      2.30
use       2.34
sound     2.39
dtype: float64

highest IDF (rare, very specific):
able control            8.63
able create             8.63
able read               8.63
able tune               8.63
able tweak              8.63
absolutely wonderful    8.63
ac amplug               8.63
ac fender               8.63
dtype: float64
